In [8]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent

import os
import base64
import json

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

print("Environment loaded successfully!")

Environment loaded successfully!


In [9]:
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    max_tokens=800
)

In [10]:
image_path = "blood_work.png"

if not os.path.exists(image_path):
    raise FileNotFoundError(
        f"Image not found: {image_path}. "
        "Place blood_work.png in your notebook's working directory."
    )

with open(image_path, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

print("Blood report image loaded successfully!")

Blood report image loaded successfully!


In [12]:
extraction_prompt = """
Read the blood test report in the image.

Extract the readable tests with:
- Test name
- Result and unit
- Reference range shown on the report
- Whether the report marks the result low, normal, or high

Prioritize glucose and cholesterol-related tests, but include other
readable results.

Do not invent missing values or reference ranges.
Mention unreadable values and do not diagnose a disease.

Keep the response concise and structured.
"""

message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{image_b64}"
            }
        },
        {
            "type": "text",
            "text": extraction_prompt
        }
    ]
)

response = llm.invoke([message])

blood_report_results = response.content

print("BLOOD REPORT ANALYSIS")
print("=" * 50)
print(blood_report_results)

BLOOD REPORT ANALYSIS
Based on the provided image, here are the extracted test results. Note that the report does not explicitly mark results as "low," "normal," or "high" with text or arrows; the status is determined by comparing the result to the provided reference range.

**Patient:** Rajesh Sharma, Age 48, Male
**Date:** May 7, 2026

### **Lipid Panel (Cholesterol-Related)**
*   **Total Cholesterol**
    *   **Result:** 238 mg/dL
    *   **Reference Range:** < 200
    *   **Status:** High (Result exceeds reference range)
*   **LDL Cholesterol**
    *   **Result:** 162 mg/dL
    *   **Reference Range:** < 100
    *   **Status:** High (Result exceeds reference range)
*   **HDL Cholesterol**
    *   **Result:** 36 mg/dL
    *   **Reference Range:** > 40
    *   **Status:** Low (Result is below reference range)
*   **Triglycerides**
    *   **Result:** 188 mg/dL
    *   **Reference Range:** < 150
    *   **Status:** High (Result exceeds reference range)

### **Metabolic Panel (Glucose-

In [13]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """
    Return general dietary suggestions for a category.

    condition must be one of:
    normal, high_cholesterol, high_sugar.

    These suggestions are educational and are not a medical diagnosis.
    """

    diet_plans = {
        "high_cholesterol": {
            "eat": [
                "Fruits",
                "Vegetables",
                "Whole grains",
                "Legumes",
                "Nuts",
                "Lean proteins"
            ],
            "limit": [
                "Foods high in saturated fat",
                "Trans fats",
                "Deep-fried foods",
                "Processed meats"
            ]
        },

        "high_sugar": {
            "eat": [
                "Non-starchy vegetables",
                "Whole grains in appropriate portions",
                "Legumes",
                "Nuts",
                "Balanced meals containing protein and fiber"
            ],
            "limit": [
                "Sugary drinks",
                "Foods with large amounts of added sugar",
                "Highly refined carbohydrates"
            ]
        },

        "normal": {
            "eat": [
                "Vegetables",
                "Fruits",
                "Whole grains",
                "Legumes",
                "Lean proteins",
                "Healthy fats"
            ],
            "limit": [
                "Excessive added sugar",
                "Highly processed foods",
                "Foods high in trans fats"
            ]
        }
    }

    condition = condition.strip().lower()

    if condition not in diet_plans:
        return {
            "error": (
                "Unknown category. Choose normal, "
                "high_cholesterol, or high_sugar."
            )
        }

    return diet_plans[condition]

In [14]:
SYSTEM_PROMPT = """
You are a cautious blood report and nutrition assistant.

You receive extracted blood test results from a previous vision-model step.

Your responsibilities:

1. Review the supplied results and their laboratory reference ranges.
2. Identify whether the available evidence suggests a cholesterol-related
   abnormality or a high blood glucose result.
3. Choose one category only when the supplied results support it:
   - normal
   - high_cholesterol
   - high_sugar
4. If a relevant abnormal result is clearly supported, call
   get_diet_recommendation with the corresponding category.
5. If the results are incomplete, unclear, or do not support a category,
   do not guess. Explain the uncertainty and recommend review by a
   qualified healthcare professional.

Important safeguards:
- Do not invent test results or reference ranges.
- Do not diagnose diabetes, cardiovascular disease, or another condition.
- Distinguish laboratory findings from a medical diagnosis.
- Do not claim that every result is normal just because no abnormality
  was identified.
- Present dietary suggestions as general educational information.
- Encourage medical advice for abnormal results and individualized needs.

In your final answer, include:
1. A concise summary of the supplied results.
2. The category selected, if supported.
3. The diet suggestions returned by the tool, if applicable.
4. Any important uncertainties or reasons to consult a clinician.
"""

diet_agent = create_agent(
    model=llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT
)

print("Diet recommendation agent initialized!")

Diet recommendation agent initialized!


In [15]:
diet_request = f"""
Here are the extracted results from the blood work report:

{blood_report_results}

Analyze only the information provided above.

Select a dietary category only if the available evidence supports it.
If the results are insufficient or ambiguous, explain why rather than
guessing.

Provide general dietary guidance and appropriate cautions.
"""

result = diet_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": diet_request
            }
        ]
    }
)

print("FINAL BLOOD WORK AND DIET ANALYSIS")
print("=" * 60)
print(result["messages"][-1].content)

FINAL BLOOD WORK AND DIET ANALYSIS
## 1. Summary of Supplied Results

**Lipid Panel (Cholesterol-Related):**
| Test | Result | Reference Range | Status |
|---|---|---|---|
| Total Cholesterol | 238 mg/dL | < 200 | **High** |
| LDL Cholesterol | 162 mg/dL | < 100 | **High** |
| HDL Cholesterol | 36 mg/dL | > 40 | **Low** |
| Triglycerides | 188 mg/dL | < 150 | **High** |

**Metabolic Panel (Glucose-Related):**
| Test | Result | Reference Range | Status |
|---|---|---|---|
| Fasting Glucose | 92 mg/dL | 70 – 99 | Normal |
| HbA1c | 5.3% | < 5.7% | Normal |

**Other Results:** Hemoglobin, Hematocrit, WBC, Platelets, Creatinine, and eGFR are all within their respective reference ranges.

**Missing:** Liver function test values are cut off and unavailable.

---

## 2. Category Selected: **high_cholesterol**

The lipid panel clearly shows a cholesterol-related abnormality:
- **Total cholesterol (238)** and **LDL (162)** are both above their reference ranges.
- **HDL (36)** is below its refer

In [16]:
for message in result["messages"]:
    print(f"\nMessage type: {type(message).__name__}")
    
    if getattr(message, "tool_calls", None):
        print("Tool calls:", message.tool_calls)

    content = getattr(message, "content", "")
    
    if content:
        print("Content:", content)


Message type: HumanMessage
Content: 
Here are the extracted results from the blood work report:

Based on the provided image, here are the extracted test results. Note that the report does not explicitly mark results as "low," "normal," or "high" with text or arrows; the status is determined by comparing the result to the provided reference range.

**Patient:** Rajesh Sharma, Age 48, Male
**Date:** May 7, 2026

### **Lipid Panel (Cholesterol-Related)**
*   **Total Cholesterol**
    *   **Result:** 238 mg/dL
    *   **Reference Range:** < 200
    *   **Status:** High (Result exceeds reference range)
*   **LDL Cholesterol**
    *   **Result:** 162 mg/dL
    *   **Reference Range:** < 100
    *   **Status:** High (Result exceeds reference range)
*   **HDL Cholesterol**
    *   **Result:** 36 mg/dL
    *   **Reference Range:** > 40
    *   **Status:** Low (Result is below reference range)
*   **Triglycerides**
    *   **Result:** 188 mg/dL
    *   **Reference Range:** < 150
    *   **Stat